In [1]:

import os
# Make sure you are in scMEDAL_for_scRNAseq dir
os.chdir("/archive/bioinformatics/DLLab/AixaAndrade/src/gitfront/dev2/scMEDAL_for_scRNAseq")
from models.models import train_model_on_named_experiment

/archive/bioinformatics/DLLab/shared/CondaEnvironments/scMEDAL/lib/python3.8/site-packages/tensorflow_addons/utils/tfa_eol_msg.py:23: UserWarning: 

TensorFlow Addons (TFA) has ended development and introduction of new features.
TFA has entered a minimal maintenance and release mode until a planned end of life in May 2024.
Please modify downstream libraries to take dependencies from other repositories in our TensorFlow community (e.g. Keras, Keras-CV, and Keras-NLP). 

For more information see: https://github.com/tensorflow/addons/issues/2807 

  warnings.warn(


## Training Models:

#### Implemented Models:
| Model Class | Model Description | 
|------------|--------------------|  
| `AE` | Autoencoder | 
| `AEC`| Autoencoder Classifier |
| `scMEDAL-FE` | Domain Adversarial Autoencoder |
| `scMEDAL-FEC`| Domain Adversarial Autoencoder Classifier |
| `scMEDAL-RE`| Domain Enhancing Autoencoder Classifier | 

#### Named Experiments:
| Valid Named Experiment | Dataset |  n_clusters | n_pred |
|------------------------|---------|-------------|--------|
| `AML`| Acute Myeloid Leukemia | 19 | 21 |
| `ASD`| Autism Spectrum Disorder | 31 | 17 | 
| `HH` | Healthy Heart | 147 | 13 | 

**Note:** If training on other datasets, the configs will need to be passed in as dictionaries to `model_kwargs` and `train_kwargs`.

`quick` is a boolean flag that can be passed to `train_kwargs` which shortens training to only 1 fold for 3 epochs.

In [ ]:
# Run "scmedalfe" and "scmedalre" models
scmedalfe_asd = train_model_on_named_experiment("scMEDAL-FE", "ASD", model_kwargs={"n_latent_dims":50}, train_kwargs={"quick":True})
scmedalre_asd = train_model_on_named_experiment("scMEDAL-RE", "ASD", model_kwargs={"n_latent_dims":50}, train_kwargs={"quick":True})


## Analyzing Models:

In [3]:

# Update output paths of the models that you just run
from utils.defaults import ASD_OUTPUTS_DIR
import os
model_folder_dict = {
    #"ae":"",
    #"ae":"",
    #"aec":"",
    "scmedalfe":"run_crossval_loss_gen_weight-1_loss_recon_weight-4000_loss_class_weight-1_n_latent_dims-50_layer_units-512-132_use_batch_norm-True_scaling-min_max_model_type-scmedalfe_batch_size-512_epochs-500_patience-30_sample_size-10000_2025-07-08_11-46",
    #"scmedalfec":"",
    "scmedalre":"run_crossval_loss_recon_weight-110.0_loss_latent_cluster_weight-0.1_n_latent_dims-50_layer_units-512-132_kl_weight-0.0_scaling-min_max_batch_size-512_epochs-500_patience-30_sample_size-10000_2025-07-08_11-48",
}
model_paths = {k:os.path.join(ASD_OUTPUTS_DIR, "latent_space", "log_transformed_2916hvggenes",k, v) for k, v in model_folder_dict.items()}      

In [ ]:
# Run MEC on scmedalfe + scmedalre, target: 17 celltypes 
mec_asd = train_model_on_named_experiment("MEC", "ASD", 
                                    train_kwargs={"quick":True, "results_path_dict":model_paths, },
                                    model_kwargs={"n_pred":17,"models_list":list(model_paths.keys()), #"get_pca":True,
                                                    #"latent_keys_config":{"fe_latent":"scmedalfe_latent", "re_latent":"scmedalre_latent"}}
                                                    "latent_keys_config":{"fe_latent":"scmedalfe_latent", "re_latent":"scmedalre_latent"}})
                                                # "latent_keys_config":{"fe_latent":"X_pca", "re_latent":"re_latent"}}

In [5]:
# Build the analysis object once
from analysis.analysis import ASDAnalysis as asd

asd = asd(
    model_result_folder_dict = model_folder_dict,
    analysis_name            = "ASD_default"     # optional tag for output folders
)

# Compile clustering scores
# Note: if you only run 1 fold, you will see NAN in CI entries
res= asd.clustering_scores(model_folder_dict,dataset_type="test")
res

Directory created or already exists: /endosome/archive/bioinformatics/DLLab/src/AixaAndrade/gitfront/dev2/scMEDAL_for_scRNAseq/outputs/ASD/compare_models/log_transformed_2916hvggenes/ASD_default

Computed aggregated scores DataFrame
Computed Confidence interval results for sample size: 10000, saved to: /endosome/archive/bioinformatics/DLLab/src/AixaAndrade/gitfront/dev2/scMEDAL_for_scRNAseq/outputs/ASD/compare_models/log_transformed_2916hvggenes/ASD_default
 


batch                                                        \
              1/db                       ch                   silhouette   
              mean CI_lower CI_upper   mean CI_lower CI_upper       mean   
dataset_type                                                               
scmedalfe     0.06      NaN      NaN  21.41      NaN      NaN      -0.17   
scmedalre     0.06      NaN      NaN  33.96      NaN      NaN      -0.39   

                               celltype                                       \
                                   1/db                          ch            
             CI_lower CI_upper     mean CI_lower CI_upper      mean CI_lower   
dataset_type                                                                   
scmedalfe         NaN      NaN     0.50      NaN      NaN   4419.89      NaN   
scmedalre         NaN      NaN     0.07      NaN      NaN  10623.31      NaN   

                                                    
                      silhouette                    
             CI_upper       mean CI_lower CI_upper  
dataset_type                                        
scmedalfe         NaN       0.25      NaN      NaN  
scmedalre         NaN       0.03      NaN      NaN

In [ ]:
# Run genomaps on a subsample input+scmedalfe recon +scmedal re recon for different batches
dict_batches = {
    "donor_5531": "ASD",
    "donor_5945": "ASD",
    "donor_5419": "ASD",
    "donor_6032": "control",
    "donor_5242": "control",
    "donor_5976": "control",
}
print("dict_batches:", dict_batches)

batches = [
    "donor_5531",
    "donor_5945",
    "donor_5419",
    "donor_6032",
    "donor_5242",
    "donor_5976",
]
# for this genomap config, you need to compute first the reconstructions from scMEDAL-FE and scMEDAL-RE
selected_models = {k: model_folder_dict[k] for k in ("scmedalfe", "scmedalre")}
gfd = asd.genomap(selected_models,
                            n_batches = 31,
                            num_iter = 2, # for quick test, otherwise 100
                            cell_id_col = "cell",
                            gene_index_col = "gene_ids",
                            celltype = ["L2/3"],
                            batches=batches,
                            models=['scmedalre'],#if add_inputs_fe=True-> scmedalfe+ inputs are used for genomap creation by default, no need to add the to the list, 
                            types=["train"], 
                            splits=[1],
                            add_inputs_fe= True,
                            extra_recon = "fe",
                            min_val = -2,
                            max_val = 6 )

In [ ]:
# Plot umaps
processors = asd.umap(model_folder_dict, types=["train"], splits=[1])
processors